## Structured Output


Models can be requested to provide their response  in a format  matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing.LangChain supports multiple schema types and methods for enforcing structured output 

## Pydantic

Pydantic models provide the richest feature set with field validation,descriptions,and nested structures.

In [40]:

import os
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = init_chat_model("google_genai:gemini-3.5-flash-lite")


In [43]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This  year  the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="the movie ratings out of 10")

In [22]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.7.7', 'langchain': '1.4.10', 'langchain-openai': '1.7.4'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002A498BB6660>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002A498BB6BA0>, root_client=<openai.OpenAI object at 0x000002A498BB5E80>, root_async_client=<openai.AsyncOpenAI object at 0x000002A498BB6900>, model_name='@cf/zai-org/glm-4.7-flash', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.cloudflare.com/client/v4/accounts/77c0b9ccbbd8551d6f35942a8782a46f/ai/v1', stream_chunk_timeout=120.0), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie

In [42]:
model.invoke("Provide deatils about the movie Bahubali")

AIMessage(content=[{'type': 'text', 'text': '**Baahubali** is a two-part Indian epic action franchise directed by **S.S. Rajamouli** and produced by Shobu Yarlagadda and Prasad Devineni under the banner of Arka Media Works. Originally filmed in both Telugu and Tamil, the films were dubbed in Hindi, Malayalam, and several other international languages. \n\nUpon release, the films shattered box office records, became a global cultural phenomenon, and redefined the scale and cinematic potential of Indian cinema—particularly Tollywood (the Telugu film industry).\n\nHere are the detailed breakdowns of the two films that make up the franchise:\n\n---\n\n### 1. Baahubali: The Beginning (2015)\n* **Release Date:** July 10, 2015\n* **Runtime:** 159 minutes\n* **Box Office:** Approx. ₹650 crore globally\n\n#### **Plot Summary:**\nThe film begins in the fictional kingdom of **Mahishmati** in ancient India. An injured woman carrying an infant is hunted by soldiers. She sacrifices her life to save 

In [23]:
model_with_structure.invoke("Provide details about the movie-Bahubali ")

Movie(title='Baahubali: The Beginning and The Conclusion', year=2015, director='S.S. Rajamouli', rating=8.2)

### Message output  alongside parsed structure

In [26]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title:str = Field(..., description="The title of the movie")
    year:int = Field(...,description="The year the movie was released")
    director:str=Field(...,description="the director of the movie")
    rating: float = Field(...,description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Bahubali")
response

{'raw': AIMessage(content='{ "title": "Baahubali: The Epic Indian Film Series", "year": 2015, "director": "S.S. Rajamouli", "rating": 8.2\n}', additional_kwargs={'parsed': Movie(title='Baahubali: The Epic Indian Film Series', year=2015, director='S.S. Rajamouli', rating=8.2), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1032, 'prompt_tokens': 13, 'total_tokens': 1045, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'neurons': 37.636300000000006}, 'model_provider': 'openai', 'model_name': '@cf/zai-org/glm-4.7-flash', 'system_fingerprint': None, 'id': 'id-1794987760293', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ed9b-50c0-75aa-a2a8-b18a7046cd98-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 1032, 'total_tokens': 1045, 'input_token_details': {'cache_read': 0}, 'output_to

### Nested structure


In [47]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    generes:list[str]
    budget:float | None = Field(None,description="Budget  in millions USD")
   
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Bahubali")
response

MovieDetails(title='Baahubali: The Beginning', year=2015, cast=[Actor(name='Prabhas', role='Amarendra Baahubali and Shivudu / Mahendra Baahubali'), Actor(name='Rana Daggubati', role='Bhallaladeva'), Actor(name='Anushka Shetty', role='Devasena'), Actor(name='Tamannaah', role='Avanthika'), Actor(name='Ramya Krishnan', role='Sivagami'), Actor(name='Sathyaraj', role='Kattappa')], generes=['Action', 'Epic', 'Fantasy', 'Drama'], budget=40.0)

## TypedDict

TypedDict provides a simpler alternative using Python built-in typing,ideal when you don't need runtime validation.

In [41]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating: Annotated[float,...,"The movie's rating out of 10"]


model_withtypedict = model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the  movie Spider-Man: Brand New Day")
response


{'title': 'Spider-Man: Brand New Day',
 'year': 2026,
 'director': 'Destin Daniel Cretton',
 'rating': 7.5}

In [48]:
from pydantic import BaseModel,Field

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    generes:list[str]
    budget:float | None = Field(None,description="Budget  in millions USD")
   
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'}],
 'generes': ['Action', 'Sci-Fi', 'Superhero'],
 'budget': 220000000}

In [49]:
model.profile

{'name': 'Gemini 3.5 Flash Lite',
 'release_date': '2026-07-21',
 'last_updated': '2026-07-21',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True,
 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'],
 'reasoning_effort_default': 'minimal'}

## DataClasses

A data class is a class typically containng mainly data, although there aren't really any restrictions.You create it using  the @dataclass decorator

In [50]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")



In [ ]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name:str = Field(description="The name of the prerson")
    email:str = Field(description="The email address of the person")
    phone:str = Field(description="The phone number of the prerson")

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact  info from: John Doe,jhon@example.com,(555)123-4567"}]
})

print(result["structured_response"])
#ContactInfo(name='John Doe', email='john@example.com',phone='(555)123-4567')


name='John Doe' email='jhon@example.com' phone='(555)123-4567'


In [55]:
result

{'messages': [HumanMessage(content='Extract contact  info from: John Doe,jhon@example.com,(555)123-4567', additional_kwargs={}, response_metadata={}, id='4f5f4f51-75ad-4cea-9e0f-443d9b3731a5'),
  AIMessage(content=[{'type': 'text', 'text': '{"name":"John Doe","email":"jhon@example.com","phone":"(555)123-4567"}', 'extras': {'signature': 'EpkLCpYLAWkUfROSZMr2Tu3uBp42dtH9ESgXBSJNDLsaAYGK3iKLDCC6I51pDi6zxwvuW4JraXi3agBEla61mXvXhBfJED1nHrlPfXdnPtgn08x3hkxBn0TYooHFQdluBX9E3UVgLYB/wSy/U05QgFfQv0mYDEaX/TFwDa7P3keEkOGuBTMSGgZxPe1L1b4IyNQ8vOkgRBSuYlhfbHkC/EVBmHtibuqItjBDORFIygzoXMECkbaXCtLY4qe+/H2a4aRvlnUxXKUNKyJ9Td1ViR7VuZEQlp3haalTShwSIc6pyLGNILY39iUba0amC1sy8DJBv+2G52DmrUhj3WQiJ1Pw8ZmmSAc04CIZfpcsddCciKFQR9IflxsPbA4RpEkZeBacpMAW7VD+SrFJoQZAz1vGOUjTCGMTfWf250f1tXFnkkFFQqp04BY/aDzBa6YWXEeELA/PaIvgpXw3WKs93gdPAfDnOnZAPaqK3R1TUMcBPnX6Hgyb6McXOeKHadgJ4k3ZJ2aVTTKupGRqJ0Ge2dkX61HXOD6dDOgOT3XiiG6SvxB8N5WwBkNYQd6OjL1DXavwmMKSbRAWt5rdQm+FED1CdxrdqamYd09aoUwME3c3XI0vl20oMM7P6D7eT//tL/cJu6qbediQHhD5yULHT

In [59]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name:str = Field(description="The name of the prerson")
    email:str = Field(description="The email address of the person")
    phone:str = Field(description="The phone number of the prerson")

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact  info from: John Doe,jhon@example.com,(555)123-4567"}]
})

print(result["structured_response"])
#ContactInfo(name='John Doe', email='john@example.com',phone='(555)123-4567')


{'name': 'John Doe', 'email': 'jhon@example.com', 'phone': '(555)123-4567'}


In [60]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name:str # The name of the person 
    email:str # The email address of the person 
    phone:str #The phone number of the person

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    response_format=ContactInfo
)

result = agent.invoke ({
    "messages":[{"role":"user","content":"Extract  contact info from : JohnDoe, john@example.com,(555) 123-4567"}]
})

result["structured_response"]



ContactInfo(name='JohnDoe', email='john@example.com', phone='(555) 123-4567')